In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader

data = torch.load('data/lichess_processed_dataset.pt', weights_only=True)
tokens_tensor = data['tokens']
moves_tensor  = data['moves']
values_tensor = data['values']
print(f"Loaded {tokens_tensor.size(0)} positions from lichess_processed_dataset.pt")
print(f"Tokens: {tokens_tensor.shape} | Moves: {moves_tensor.shape} | Values: {values_tensor.shape}")

dataset = TensorDataset(tokens_tensor, moves_tensor, values_tensor)
val_fraction = 0.1
val_size = int(len(dataset) * val_fraction)
train_size = len(dataset) - val_size
split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=split_generator
)
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Optimized DataLoader settings:
# - persistent_workers=True keeps workers alive between epochs (avoids respawn overhead)
# - prefetch_factor=4 preloads more batches for smoother GPU feeding
train_loader = DataLoader(train_dataset, batch_size=870, shuffle=True,
                          pin_memory=True, num_workers=6, drop_last=False,
                          persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_dataset,   batch_size=870, shuffle=False,
                          pin_memory=True, num_workers=6, drop_last=False,
                          persistent_workers=True, prefetch_factor=4)

Loaded 12459170 positions from lichess_processed_dataset.pt
Tokens: torch.Size([12459170, 65]) | Moves: torch.Size([12459170]) | Values: torch.Size([12459170])
Train size: 11213253 | Val size: 1245917


In [4]:
import math
import torch
import torch.nn as nn

class ChessTransformer(nn.Module):
    def __init__(self, vocab_size=15, d_model=512, nhead=8, num_layers=8, dropout=0.1, head_dim=256):
        super().__init__()
        
        # 1. Input Embedding & Dropout
        # vocab_size is 15: 0(empty), 1-6(White pieces), 7-12(Black pieces), 13(White turn), 14(Black turn)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        
        # 2. Positional Encoding
        # Sequence length is 65: 64 squares + 1 turn token
        self.pos_encoder = nn.Parameter(torch.randn(1, 65, d_model) * 0.02)
        
        # 3. Transformer Backbone with Dropout
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. The Value Head (Mean Pooling over all 65 tokens)
        self.value_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
            nn.Tanh()
        )
        
        # 5. The Policy Head (Bilinear Interaction over first 64 tokens only)
        self.head_dim = head_dim
        self.from_proj = nn.Linear(d_model, head_dim)
        self.to_proj = nn.Linear(d_model, head_dim)
        self.logit_scale = 1.0 / math.sqrt(head_dim)

    def forward(self, x, legal_move_mask=None):
        # x shape: (Batch, 65)
        
        # Step 1 & 2: Embed, add pos encoding, and apply dropout
        x = self.embedding(x) + self.pos_encoder
        x = self.emb_dropout(x)
        
        # Step 3: Transformer
        x = self.transformer(x)  # Output shape: (Batch, 65, d_model)
        
        # Step 4: Value Head via Mean Pooling (all 65 tokens)
        pooled_state = x.mean(dim=1)  # Shape: (Batch, d_model)
        value = self.value_head(pooled_state).squeeze(-1)  # Shape: (Batch,)
        
        # Step 5: Policy Head via Scaled Bilinear Dot Product (first 64 tokens only)
        x_squares = x[:, :64, :]  # Shape: (Batch, 64, d_model) - exclude turn token
        from_feats = self.from_proj(x_squares)  # Shape: (Batch, 64, head_dim)
        to_feats = self.to_proj(x_squares)      # Shape: (Batch, 64, head_dim)
        
        # Scaled dot-product keeps softmax temperature reasonable at init
        policy_logits = torch.bmm(from_feats, to_feats.transpose(1, 2)) * self.logit_scale
        policy_logits = policy_logits.view(x.size(0), 4096)
        
        # Step 6: Legal Move Masking
        if legal_move_mask is not None:
            policy_logits = policy_logits.masked_fill(~legal_move_mask, float('-inf'))
            
        return policy_logits, value

In [5]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp.autocast_mode import autocast
from torch.optim.lr_scheduler import OneCycleLR

# Enable cuDNN autotuning for fixed input sizes
torch.backends.cudnn.benchmark = True

# Create models directory
os.makedirs('models', exist_ok=True)

# 1. Initialize Device, Model, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = ChessTransformer().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Note: torch.compile() requires Triton which is Linux-only
# Skipping on Windows - other optimizations still apply

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
policy_criterion = nn.CrossEntropyLoss()
value_criterion = nn.MSELoss()

VALUE_LOSS_WEIGHT = 0.5

epochs = 15
num_batches = len(train_loader)
print(f"Train batches: {num_batches} | Val batches: {len(val_loader)} | Epochs: {epochs}")

# OneCycleLR: ramps LR up then down, often improves convergence
# Reduced max_lr for larger model width (d_model=512)
scheduler = OneCycleLR(optimizer, max_lr=5e-4, epochs=epochs, steps_per_epoch=num_batches)
print(f"Using OneCycleLR scheduler (max_lr=5e-4)\n")

def train():
    best_val_loss = float('inf')
    best_val_acc = 0.0

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        total_policy_loss = 0.0
        total_value_loss = 0.0
        correct_moves = 0
        total_positions = 0
        
        for batch_idx, (tokens, target_moves, target_values) in enumerate(train_loader):
            tokens = tokens.to(device, non_blocking=True)
            target_moves = target_moves.to(device, non_blocking=True)
            target_values = target_values.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                policy_logits, predicted_values = model(tokens)
                policy_loss = policy_criterion(policy_logits, target_moves)
                value_loss = value_criterion(predicted_values, target_values)
                batch_loss = policy_loss + VALUE_LOSS_WEIGHT * value_loss
                
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += batch_loss.item()
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            
            predictions = torch.argmax(policy_logits, dim=1)
            correct_moves += (predictions == target_moves).sum().item()
            total_positions += tokens.size(0)
            
            if batch_idx % 50 == 0:
                current_acc = (correct_moves / total_positions) * 100
                current_lr = scheduler.get_last_lr()[0]
                print(
                    f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{num_batches} | "
                    f"Loss: {batch_loss.item():.4f} "
                    f"(P: {policy_loss.item():.4f}, V: {value_loss.item():.4f}) | "
                    f"Acc: {current_acc:.2f}% | LR: {current_lr:.2e}"
                )
        
        train_time = time.time() - epoch_start
        train_acc = (correct_moves / total_positions) * 100
        train_avg_loss = total_loss / num_batches
        train_avg_policy = total_policy_loss / num_batches
        train_avg_value = total_value_loss / num_batches
        
        # --- Validate ---
        model.eval()
        val_start = time.time()
        val_total_loss = 0.0
        val_total_policy_loss = 0.0
        val_total_value_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for tokens, target_moves, target_values in val_loader:
                tokens = tokens.to(device, non_blocking=True)
                target_moves = target_moves.to(device, non_blocking=True)
                target_values = target_values.to(device, non_blocking=True)
                
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    policy_logits, predicted_values = model(tokens)
                    policy_loss = policy_criterion(policy_logits, target_moves)
                    value_loss = value_criterion(predicted_values, target_values)
                    batch_loss = policy_loss + VALUE_LOSS_WEIGHT * value_loss
                
                val_total_loss += batch_loss.item()
                val_total_policy_loss += policy_loss.item()
                val_total_value_loss += value_loss.item()
                val_correct += (torch.argmax(policy_logits, dim=1) == target_moves).sum().item()
                val_total += tokens.size(0)
        
        val_time = time.time() - val_start
        val_acc = (val_correct / val_total) * 100
        val_avg_loss = val_total_loss / len(val_loader)
        val_avg_policy = val_total_policy_loss / len(val_loader)
        val_avg_value = val_total_value_loss / len(val_loader)
        
        print(
            f"--- Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_avg_loss:.4f} (P: {train_avg_policy:.4f}, V: {train_avg_value:.4f}) Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_avg_loss:.4f} (P: {val_avg_policy:.4f}, V: {val_avg_value:.4f}) Acc: {val_acc:.2f}% | "
            f"Time: {train_time:.1f}s train + {val_time:.1f}s val ---"
        )
        
        # --- Save best model with accuracy and param count in filename ---
        if val_avg_loss < best_val_loss:
            best_val_loss = val_avg_loss
            best_val_acc = val_acc
            
            # Format: chess_transformer_{params}_{accuracy}pct.pt
            params_str = f"{num_params // 1000}k" if num_params < 1_000_000 else f"{num_params / 1_000_000:.1f}M"
            checkpoint_path = f"models/guofish_{params_str}{val_acc:.1f}p.pt"
            
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_avg_loss,
                'val_acc': val_acc,
                'num_params': num_params,
            }, checkpoint_path)
            print(f"New best val loss: {val_avg_loss:.4f} — saved to {checkpoint_path}\n")
        else:
            print(f"No improvement (best val loss: {best_val_loss:.4f})\n")

if __name__ == "__main__":
    train()

Training on device: cuda
GPU: NVIDIA GeForce RTX 5070
Model parameters: 25,785,857


c:\Users\Ethan Guo\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Train batches: 12889 | Val batches: 1433 | Epochs: 15
Using OneCycleLR scheduler (max_lr=5e-4)

Epoch 1/15 | Batch 0/12889 | Loss: 10.9873 (P: 10.5280, V: 0.9185) | Acc: 0.00% | LR: 2.00e-05
Epoch 1/15 | Batch 50/12889 | Loss: 6.8268 (P: 6.3829, V: 0.8878) | Acc: 0.47% | LR: 2.00e-05


KeyboardInterrupt: 